# LocalCodeGraph: free-Colab T4 repair pilot

Choose **Runtime → Change runtime type → T4 GPU**. This notebook uses **Gemma 4 E2B with bitsandbytes NF4 and FP16 compute**, not the 31B model or official W4A16 QAT weights. It does not require vLLM, an A100, paid compute or a hosted API.

This configuration targets T4 memory; actual fit/speed still require your runtime test. CPU CI checks the notebook/controller only. The 23-task, paired BM25-versus-graph pilot generates patches; separate test-based grading is required before reporting repair success.

Run cells in order. Each attempt is checkpointed. You can run the download cell after an interruption to save partial work. Free Colab resource availability and session length vary.


In [ ]:
import json, os, pathlib, shutil, subprocess, sys
ROOT = pathlib.Path("/content/LocalCodeGraph")
VENV = pathlib.Path("/content/lcg-t4-env")
MODEL_DIR = pathlib.Path("/content/gemma-4-e2b")
INPUTS = pathlib.Path("/content/lcg-t4-inputs")
OUTPUT = pathlib.Path("/content/lcg-t4-run")
MODEL = "google/gemma-4-E2B-it"
RESUME = False  # True only to continue this exact run; completed/errors are never retried.
gpu = subprocess.check_output(
    ["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader,nounits"], text=True)
print(gpu)
if not any(int(line.rsplit(",", 1)[1]) >= 14000 for line in gpu.strip().splitlines()):
    raise RuntimeError("Select a T4 GPU runtime with approximately 15 GiB usable VRAM.")
if shutil.disk_usage("/content").free < 30 * 1024**3 and not MODEL_DIR.exists():
    raise RuntimeError("Need 30 GiB free disk for initial installation and base-model download.")
if OUTPUT.exists() and not RESUME:
    raise RuntimeError("Output already exists. Preserve it or set RESUME=True for this exact run.")


## Install without replacing Colab's CUDA/PyTorch stack

A virtual environment reuses the installed Colab packages. New inference dependencies stay in that environment. The exact resolved versions are saved with the results. No GPU package is installed into the notebook kernel.


In [ ]:
def checked(command):
    result = subprocess.run(command, capture_output=True, text=True)
    print(result.stdout, flush=True)
    if result.returncode:
        print(result.stderr, flush=True)
        raise RuntimeError("Setup command failed; complete diagnostics are printed above.")
    return result

if not ROOT.exists():
    checked(["git", "clone", "https://github.com/eluan216/LocalCodeGraph.git", str(ROOT)])
else:
    # Bring an earlier clone forward without resetting local edits.
    checked(["git", "-C", str(ROOT), "pull", "--ff-only"])
REPO_COMMIT = subprocess.check_output(["git", "-C", str(ROOT), "rev-parse", "HEAD"], text=True).strip()
checked([sys.executable, str(ROOT / "colab/bootstrap_env.py"), "--destination", str(VENV)])
PYTHON = str(VENV / "bin/python")
checked([PYTHON, "-m", "pip", "install", "--upgrade",
    "-e", str(ROOT), "transformers", "accelerate", "bitsandbytes", "huggingface_hub",
    "pyarrow==26.0.1", "unidiff==1.0.1"])
print("Toolkit commit:", REPO_COMMIT)


## Prepare paired inputs

Only issue text and pre-fix context enter the generator. The frozen dataset has 23 public development tasks. Existing inputs are checksum-validated by the controller; do not change them between arms.


In [ ]:
if not INPUTS.exists():
    subprocess.run([PYTHON, str(ROOT / "benchmarks/prepare_repair_inputs.py"),
                    "--output-dir", str(INPUTS)], check=True)
manifest = json.loads((INPUTS / "manifest.json").read_text())
assert manifest["tasks"] == 23
print("Input SHA256:", manifest["inputs_sha256"])


## Download E2B at one immutable revision

Quantization happens during model loading. The download contains original base weights, so the disk requirement is larger than final GPU storage. If authentication is needed, use Colab Secrets named HF_TOKEN; do not paste a token into code. Resume reuses the previously recorded revision.


In [ ]:
download_env = os.environ.copy()
try:
    from google.colab import userdata
    token = userdata.get("HF_TOKEN")
    if token:
        download_env["HF_TOKEN"] = token
except Exception:
    pass
download_code = r'''
import json, sys
from pathlib import Path
from huggingface_hub import HfApi, snapshot_download
model, destination = sys.argv[1:]
marker = Path(destination) / "lcg-revision.json"
revision = json.loads(marker.read_text())["revision"] if marker.exists() else HfApi().model_info(model).sha
snapshot_download(model, revision=revision, local_dir=destination,
    allow_patterns=["*.json", "*.jinja", "*.safetensors", "*.model", "*.txt", "README.md", "LICENSE*"])
marker.write_text(json.dumps({"model": model, "revision": revision}))
print("Model revision:", revision)
'''
subprocess.run([PYTHON, "-c", download_code, MODEL, str(MODEL_DIR)],
               env=download_env, check=True)
download_env.pop("HF_TOKEN", None)
MODEL_REVISION = json.loads((MODEL_DIR / "lcg-revision.json").read_text())["revision"]


## Generate E2B NF4 patches locally

No server is started. The T4 process uses NF4 linear layers, FP16 compute, SDPA attention, a 4,096-token input cap and a 1,024-token output cap. The same issue prefix and instructions are used in both arms. Input trimming and raw responses are recorded.

A first load/generation failure is not a repair result. Keep its error details and do not substitute a different model without labeling a new experiment. Resume skips recorded successes **and errors** to prevent best-of selection.


In [ ]:
command = [PYTHON, "-m", "localcodegraph.repair_t4",
    "--inputs", str(INPUTS / "inputs.jsonl"),
    "--manifest", str(INPUTS / "manifest.json"),
    "--model-dir", str(MODEL_DIR), "--model-revision", MODEL_REVISION,
    "--output-dir", str(OUTPUT)]
if RESUME:
    command.append("--resume")
inference_env = os.environ.copy()
inference_env.update(HF_HUB_OFFLINE="1", TRANSFORMERS_OFFLINE="1")
subprocess.run(command, env=inference_env, check=True)
print("Generation finished. Patch statuses are not test-based repair scores.")


## Save outputs—even after an interrupted generation cell

Run this cell whenever you want to save partial work. Attempts are exported after every completion. A Colab VM deletion can erase files, so download before leaving. Model weights are not included in the ZIP. The exact gold-free inputs and upstream notices are included.


In [ ]:
if not OUTPUT.exists():
    raise RuntimeError("No run output exists yet.")
(OUTPUT / "environment.txt").write_text(
    subprocess.check_output([PYTHON, "-m", "pip", "freeze"], text=True))
(OUTPUT / "hardware.txt").write_text(subprocess.check_output(["nvidia-smi"], text=True))
(OUTPUT / "toolkit-commit.txt").write_text(REPO_COMMIT + "\n")
shutil.copytree(INPUTS, OUTPUT / "gold-free-inputs", dirs_exist_ok=True)
archive = shutil.make_archive("/content/localcodegraph-t4-e2b-repair", "zip", OUTPUT)
from google.colab import files
files.download(archive)


## Test-based grading

Use the exported predictions.bm25.jsonl and predictions.graph.jsonl on a separate Docker-capable host with the official SWE-bench grader and the dev split. Keep E2B NF4 results separate from the archived 31B QAT pilot. If the notebook was interrupted, both arms must have all 23 planned attempts before a complete paired result can be claimed.

[Setup and grading guide](https://github.com/eluan216/LocalCodeGraph/blob/main/colab/README.md) · [T4 protocol](https://github.com/eluan216/LocalCodeGraph/blob/main/benchmarks/T4_REPAIR_PROTOCOL.md) · [Bitsandbytes hardware support](https://huggingface.co/docs/transformers/quantization/bitsandbytes)
